# EDA: Классификация и граф контрагентов

*Разведочный анализ очищенного датасета транзакций.*

**Что делаем по заданию (Часть 1.1):**
- размер, период, число уникальных контрагентов
- топ-20 контрагентов по обороту (входящие / исходящие отдельно)
- распределение сумм (log-scale) и помесячная динамика
- минимум 3 аномалии с пояснением

Вся очистка вынесена в `src/cleaning.py` этот ноутбук только вызывает функции.

In [3]:
import sys
from pathlib import Path

# из notebooks/ поднимаемся в корень проекта, подключаем src/
ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import cleaning as c

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Загрузка и очистка

In [5]:
raw = c.load_raw(ROOT / "data" / "transactions.csv")
clean = c.clean_transactions(ROOT / "data" / "transactions.csv")
print("сырых строк:", len(raw), "| после очистки:", len(clean))

сырых строк: 80800 | после очистки: 80000


### Таблица «было / стало»

In [7]:
c.build_before_after_table(raw, clean)

,Метрика,Было,Стало
0,Строк всего,80800,80000
1,Уникальных операций:,80000,None
2,Доля валидных ID (контр. сумма РК),98.6%,99.5%
3,Пропуски в дате,0.0%,0.0%
4,Пропуски в description,1.0%,1.0%


## 2. Размер, период, контрагенты

In [59]:
period_start = clean["date_clean"].min().date()
period_end = clean['date_clean'].max().date()
# валидные ID отправителей и получателей
all_ids = pd.concat([clean.loc[clean["sender_valid"], "sender_id_clean"],
    clean.loc[clean["receiver_valid"], "receiver_id_clean"]]
)

n_counterparties = all_ids.nunique()
print(f"Операций:  {len(clean):,}")
print(f"Период:  {period_start} - {period_end}")
print(f"Уникальных контрагентов:  {n_counterparties:}")
print(f"Сумма оборота, всего:  {clean['amount_kzt'].sum():,.0f} тенге")

Операций:  80,000
Период:  2024-01-01 - 2026-04-30
Уникальных контрагентов:  5000
Сумма оборота, всего:  164,258,221,566 тенге


## 3. Топ-20 контрагентов по обороту

### 3.1 По исходящим (кто больше всех платит)

In [61]:
top_out = (clean[clean["sender_valid"]]
    .groupby("sender_id_clean")["amount_kzt"].sum()
    .nlargest(20).rename("исходящий_оборот"))
top_out.to_frame()

,исходящий_оборот
sender_id_clean,
720221554469,"765,219,632.69"
580811600403,"730,443,480.93"
310909691346,"722,426,615.65"
181226594058,"704,139,580.96"
170915651329,"699,218,914.93"
401110643824,"688,859,601.18"
140512557513,"684,582,177.52"
941101594315,"666,485,775.29"
970913552723,"639,378,908.22"


### 3.2 По входящим (кто больше всех получает)

In [63]:
top_in = (clean[clean["receiver_valid"]]
.groupby("receiver_id_clean")["amount_kzt"].sum()
    .nlargest(20).rename("входящий_оборот"))
top_in.to_frame()

,входящий_оборот
receiver_id_clean,
850603506938,"371,222,531.02"
711020691210,"361,082,258.84"
410517502025,"334,064,919.03"
551211641020,"333,971,316.50"
730424495604,"320,260,000.23"
850623490114,"295,327,655.22"
121125400846,"294,152,154.00"
990508519961,"274,385,319.50"
861228440096,"269,006,165.98"
